## 任务
从上海证券交易所抓取**农业银行**4月份所有交易日的的**股票数据**

## 字段
日期，开盘(元)，收盘(元)，涨跌幅，最高(元)，最低(元)，成交量(手)，成交金额(万元)，换手率

# 格式
- 生成csv格式
- 以markdown返回

# 其他要求
**不要使用千分位**的字段： 成交量(手), 成交金额(万元) 


In [4]:
import pandas as pd


df = pd.read_csv('shex-601288-2604.csv', encoding='utf-8')


df

,日期,开盘(元),收盘(元),涨跌幅(%),最高(元),最低(元),成交量(手),成交金额(万元),换手率(%)
0,2026-04-30,6.92,6.92,-0.29,7.00,6.86,3607437,249902.22,0.11
1,2026-04-29,6.97,6.94,-0.57,7.00,6.85,3185611,220414.19,0.10
2,2026-04-28,6.90,6.98,0.87,6.99,6.87,2609718,180920.48,0.08
3,2026-04-27,7.02,6.92,-1.70,7.07,6.90,2422166,168509.41,0.08
4,2026-04-24,7.03,7.04,0.14,7.06,6.98,1961004,137847.80,0.06
5,2026-04-23,7.07,7.03,-0.57,7.11,7.02,2496937,176318.56,0.08
6,2026-04-22,7.18,7.07,-1.53,7.18,7.02,2737146,193867.89,0.09
7,2026-04-21,7.20,7.18,-0.14,7.29,7.15,2544182,183454.27,0.08
8,2026-04-20,7.05,7.19,1.84,7.22,7.04,3201580,229420.23,0.10
9,2026-04-17,7.00,7.06,0.86,7.12,6.98,3558279,251383.09,0.11



安装playwright
pip install playwright
安装浏览器驱动
playwright install chromium
如果需要其他依赖
pip install asyncio

In [ ]:


import asyncio
from playwright.async_api import async_playwright
import json
import logging

async def fetch_with_stealth_approach():
    """
    使用更隐蔽的方式抓取数据（模拟真实用户行为）
    """
    async with async_playwright() as p:
        # 使用更真实的启动参数
        browser = await p.chromium.launch(
            headless=False,  # 使用有头模式更容易绕过检测
            args=[
                '--disable-blink-features=AutomationControlled',
                '--disable-features=IsolateOrigins,site-per-process',
                '--no-sandbox',
                '--disable-dev-shm-usage'
            ]
        )
        
        context = await browser.new_context(
            viewport={'width': 1920, 'height': 1080},
            user_agent='Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            locale='zh-CN',
            timezone_id='Asia/Shanghai',
            extra_http_headers={
                'Accept': 'application/json, text/plain, */*',
                'Accept-Language': 'zh-CN,zh;q=0.9,en;q=0.8',
                'Referer': 'https://www.douyin.com/',
                'Origin': 'https://www.douyin.com'
            }
        )
        
        page = await context.new_page()
        
        # 先访问抖音首页，模拟真实用户行为
        logging.info("先访问抖音首页...")
        await page.goto('https://www.douyin.com/', wait_until='networkidle', timeout=30000)
        await asyncio.sleep(2)  # 等待2秒，模拟用户浏览
        
        # 然后访问API接口
        url = "https://www.iesdouyin.com/web/api/v2/hotsearch/billboard/word/"
        logging.info(f"开始请求API: {url}")
        
        try:
            # 使用fetch直接请求API
            result = await page.evaluate(f"""
                async () => {{
                    try {{
                        const response = await fetch('{url}', {{
                            method: 'GET',
                            headers: {{
                                'Accept': 'application/json',
                                'Referer': 'https://www.douyin.com/',
                                'Origin': 'https://www.douyin.com'
                            }}
                        }});
                        const data = await response.json();
                        return data;
                    }} catch(e) {{
                        return {{error: e.message}};
                    }}
                }}
            """)
            
            if result and 'error' not in result:
                logging.info("成功获取数据")
                print(json.dumps(result, ensure_ascii=False, indent=2))
                return result
            else:
                logging.error(f"获取数据失败: {result}")
                return None
                
        except Exception as e:
            logging.error(f"请求失败: {str(e)}")
            return None
        finally:
            if browser:
                await asyncio.sleep(1)
                await browser.close()


data = await fetch_with_stealth_approach()
data

In [10]:
%pip install crawl4ai

Looking in indexes: http://mirrors.aliyun.com/pypi/simple
     ---------------------------------------- 0.0/501.9 kB ? eta -:--:--
     --------------------- ---------------- 286.7/501.9 kB 5.9 MB/s eta 0:00:01
     -------------------------------------- 501.9/501.9 kB 5.2 MB/s eta 0:00:00
     ---------------------------------------- 0.0/462.9 kB ? eta -:--:--
     -------------------- ----------------- 245.8/462.9 kB 7.6 MB/s eta 0:00:01
     -------------------------------------- 462.9/462.9 kB 7.3 MB/s eta 0:00:00
     ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
     -- ------------------------------------- 0.3/3.8 MB 4.1 MB/s eta 0:00:01
     ------ --------------------------------- 0.6/3.8 MB 5.2 MB/s eta 0:00:01
     --------- ------------------------------ 0.9/3.8 MB 5.4 MB/s eta 0:00:01
     ------------ --------------------------- 1.2/3.8 MB 5.6 MB/s eta 0:00:01
     --------------- ------------------------ 1.5/3.8 MB 5.9 MB/s eta 0:00:01
     ----------


[notice] A new release of pip is available: 23.2.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import asyncio
import threading
from crawl4ai import AsyncWebCrawler
import nest_asyncio

# Apply nest_asyncio just in case
nest_asyncio.apply()

def run_crawl_sync(url):
    """
    A synchronous wrapper that sets up a clean event loop 
    in a dedicated thread.
    """
    result_container = []

    async def _crawl():
        async with AsyncWebCrawler(verbose=True) as crawler:
            res = await crawler.arun(url=url)
            result_container.append(res)

    def thread_worker():
        # Force the correct loop for Windows within this thread
        loop = asyncio.WindowsProactorEventLoopPolicy().new_event_loop()
        asyncio.set_event_loop(loop)
        loop.run_until_complete(_crawl())
        loop.close()

    thread = threading.Thread(target=thread_worker)
    thread.start()
    thread.join()
    
    return result_container[0] if result_container else None

# Usage
result = run_crawl_sync("https://www.iesdouyin.com/web/api/v2/hotsearch/billboard/word/")
if result:
    print(f"Title: {result.metadata.get('title')}")
    print(f"Markdown snippet: {result.markdown[:200]}")

[INIT].... → Crawl4AI 0.8.6 

[FETCH]... ↓ https://www.iesdouyin.com/web/api/v2/hotsearch/billboard/word/                                       |
✓ | ⏱: 0.80s 

[SCRAPE].. ◆ https://www.iesdouyin.com/web/api/v2/hotsearch/billboard/word/                                       |
✓ | ⏱: 0.00s 

[COMPLETE] ● https://www.iesdouyin.com/web/api/v2/hotsearch/billboard/word/                                       |
✓ | ⏱: 0.81s 

Title: None
Markdown snippet: 
```
{"active_time":"2026-05-01 16:51:16","extra":{"logid":"20260501165134197999529B95C6374727","now":1777625494327},"status_code":0,"word_list":[{"word":"今天是五一劳动节","hot_value":11053175,"label":3},{"w


In [26]:
import json
json_str = result.markdown.replace('\n','').replace('```','')
json_data = json.loads(json_str)
print('='*60)
print(f'updated date: {json_data["active_time"]}')
print('='*60)
for item in json_data['word_list']:
    print(f'{item["word"]} : {item["hot_value"]}')

updated date: 2026-05-01 16:51:16
今天是五一劳动节 : 11053175
如果幸福是一顿漂亮饭 : 11025368
劳作的声音是最动人的赞歌 : 11001821
江苏常晋晋级中冠决赛队员齐发声 : 10944693
各地老字号申请上桌 : 10377983
CBA季后赛广州vs广东前瞻 : 10268451
南部战区战备警巡黄岩岛画面 : 9153799
DeepSeek多模态正式发布 : 9111510
苏超球迷为江苏队团结起来了 : 9048143
日子滚烫 劳动至上 : 8971273
时代少年团演唱会 : 8861042
一周谣言鉴别 : 8739120
ENEMY二人离开无限流世界 : 8594876
为闪光的劳动者点赞 : 8537123
我也成为神兵小将 : 8478169
轮到我跳心如止水了 : 8418918
交给白鹿掌镜你就放心吧 : 8418791
我不是戏神完结 : 8323281
电影消失的人上映 : 8322436
REDRED好舞不挑曲 : 8061175
电影消失的人第89分钟全场尖叫 : 8051574
三角洲姜文翻拍挑战 : 7811872
鞠婧祎万花世界先行曲 : 7777106
76人将与凯尔特人抢七 : 7756014
我独爱平凡有力量的照片 : 7752734
和平精英仙逆动画联动 : 7742063
五一致敬努力工作的自己 : 7740286
陈思罕拍手变装帅成啥了 : 7730675
如果幸福是一碗榴莲山竹冰 : 7727490
00后对10后评价的解答 : 7714312
娜塔莎颠沛流离的一生 : 7710905
赵露思陈泽梦幻联动 : 7709312
北方人的蒜薹噩梦 : 7705788
有宝贝的假期每一帧都快乐 : 7705176
林子烨又cos了我不是戏神 : 7697523
李晨回忆跑男的十几年好好哭 : 7695897
藏族青年土豆也来模仿刘思懿了 : 7694603
此沙新剧直球表白任敏 : 7693884
明日方舟七周年 : 7687808
五一美食判官已上线 : 7684563
今天和贺峻霖锁了 : 7683954
画一个玻璃香蕉 : 7681546
这样的边伯贤我无法抵抗 : 7676143
恋与深空上线雾散了 : 7671582
三角洲古墓丽影联动上线 : 7